# 00 - Exploratory Data Analysis (EDA)
## Business Entity Resolution Challenge

This notebook performs initial exploration of the training and test data:
- Load all 6 files with proper TSV parsing
- Analyze row counts, countries, null values
- Examine ground truth statistics
- Sample true positive and hard negative pairs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.append('../src')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")

## 1. Load Data with Proper TSV Parsing

In [ ]:
# Define data paths
data_root = Path('../../../student_resource/dataset')
train_dir = data_root / 'train'
test_dir = data_root / 'test'

print(f"Data root: {data_root.absolute()}")
print(f"Train dir exists: {train_dir.exists()}")
print(f"Test dir exists: {test_dir.exists()}")

In [ ]:
# Load training data with sep='\t'
train_s1 = pd.read_csv(train_dir / 'train_source1.tsv', sep='\t')
train_s2 = pd.read_csv(train_dir / 'train_source2.tsv', sep='\t')
train_s3 = pd.read_csv(train_dir / 'train_source3.tsv', sep='\t')
train_gt = pd.read_csv(train_dir / 'train_ground_truth.tsv', sep='\t')

print("Training data loaded:")
print(f"  Source 1: {len(train_s1):,} records")
print(f"  Source 2: {len(train_s2):,} records")
print(f"  Source 3: {len(train_s3):,} records")
print(f"  Ground truth: {len(train_gt):,} records")

In [ ]:
# Load test data
test_s1 = pd.read_csv(test_dir / 'test_source1.tsv', sep='\t')
test_s2 = pd.read_csv(test_dir / 'test_source2.tsv', sep='\t')
test_s3 = pd.read_csv(test_dir / 'test_source3.tsv', sep='\t')

print("Test data loaded:")
print(f"  Source 1: {len(test_s1):,} records")
print(f"  Source 2: {len(test_s2):,} records")
print(f"  Source 3: {len(test_s3):,} records")

In [ ]:
# Verify columns are parsed correctly (should have 4 columns, not 1)
print("\nColumn verification:")
print(f"Train S1 columns: {train_s1.columns.tolist()}")
print(f"Train S1 shape: {train_s1.shape}")
print(f"\nFirst record from Train S1:")
print(train_s1.head(1).T)

## 2. Country Distribution Analysis

In [ ]:
# Combine all training data
train_s1['source'] = 'S1'
train_s2['source'] = 'S2'
train_s3['source'] = 'S3'
train_all = pd.concat([train_s1, train_s2, train_s3], ignore_index=True)

# Country distribution in training
print("Training Data - Country Distribution:")
country_train = train_all.groupby(['country', 'source']).size().unstack(fill_value=0)
print(country_train)
print(f"\nUnique countries in training: {sorted(train_all['country'].unique())}")

In [ ]:
# Combine all test data
test_s1['source'] = 'S1'
test_s2['source'] = 'S2'
test_s3['source'] = 'S3'
test_all = pd.concat([test_s1, test_s2, test_s3], ignore_index=True)

# Country distribution in test
print("\nTest Data - Country Distribution:")
country_test = test_all.groupby(['country', 'source']).size().unstack(fill_value=0)
print(country_test)
print(f"\nUnique countries in test: {sorted(test_all['country'].unique())}")

# Check for France in test
if 'France' in test_all['country'].values:
    print("\n⚠️  IMPORTANT: France appears in test data but NOT in training data!")
    print("    Must handle as open-set country problem.")

## 3. Null/Empty Value Audit

In [ ]:
def audit_nulls(df, name):
    """Audit null and empty values in dataframe."""
    print(f"\n{name}:")
    print("-" * 60)
    for col in df.columns:
        null_count = df[col].isnull().sum()
        empty_count = (df[col].astype(str).str.strip() == '').sum()
        total_missing = null_count + empty_count
        pct = 100 * total_missing / len(df)
        if total_missing > 0:
            print(f"  {col:20s}: {total_missing:6d} missing ({pct:5.2f}%)")
    
# Audit all datasets
audit_nulls(train_s1.drop('source', axis=1), "Train Source 1")
audit_nulls(train_s2.drop('source', axis=1), "Train Source 2")
audit_nulls(train_s3.drop('source', axis=1), "Train Source 3")
audit_nulls(test_s1.drop('source', axis=1), "Test Source 1")
audit_nulls(test_s2.drop('source', axis=1), "Test Source 2")
audit_nulls(test_s3.drop('source', axis=1), "Test Source 3")

## 4. Ground Truth Statistics

In [ ]:
# Parse matched_entity_ids
def parse_matches(match_str):
    """Parse comma-separated match IDs."""
    if pd.isna(match_str) or str(match_str).strip() == '':
        return []
    return [s.strip() for s in str(match_str).split(',') if s.strip()]

train_gt['matches'] = train_gt['matched_entity_ids'].apply(parse_matches)
train_gt['num_matches'] = train_gt['matches'].apply(len)

print("Ground Truth Statistics:")
print(f"  Total Source 1 entities: {len(train_gt):,}")
print(f"\nMatch count distribution:")
print(train_gt['num_matches'].value_counts().sort_index())

# Singleton percentage
singleton_pct = 100 * (train_gt['num_matches'] == 0).sum() / len(train_gt)
print(f"\n  Singletons (no matches): {(train_gt['num_matches'] == 0).sum():,} ({singleton_pct:.2f}%)")
print(f"  Has matches: {(train_gt['num_matches'] > 0).sum():,} ({100-singleton_pct:.2f}%)")

# Check for duplicate source1_entity_id
duplicates = train_gt['source1_entity_id'].duplicated().sum()
print(f"\n  Duplicate source1_entity_id: {duplicates}")
if duplicates > 0:
    print("  ⚠️  WARNING: Ground truth has duplicate S1 IDs!")

In [ ]:
# Visualize match distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
match_dist = train_gt['num_matches'].value_counts().sort_index()
axes[0].bar(match_dist.index, match_dist.values)
axes[0].set_xlabel('Number of Matches')
axes[0].set_ylabel('Count')
axes[0].set_title('Match Count Distribution')
axes[0].grid(axis='y', alpha=0.3)

# Pie chart for singleton vs non-singleton
singleton_counts = [(train_gt['num_matches'] == 0).sum(), (train_gt['num_matches'] > 0).sum()]
axes[1].pie(singleton_counts, labels=['Singletons', 'Has Matches'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Singleton vs Has Matches')

plt.tight_layout()
plt.show()

## 5. Sample True Positive Pairs

In [ ]:
# Get records with matches
has_matches = train_gt[train_gt['num_matches'] > 0].sample(min(20, len(train_gt[train_gt['num_matches'] > 0])))

print("Sample True Positive Pairs (Source 1 → Matches):")
print("=" * 100)

for idx, row in has_matches.head(10).iterrows():
    s1_id = row['source1_entity_id']
    s1_rec = train_s1[train_s1['entity_id'] == s1_id].iloc[0]
    
    print(f"\n[{s1_id}] {s1_rec['business_name']}")
    print(f"  Address: {s1_rec['business_address']}")
    print(f"  Country: {s1_rec['country']}")
    print(f"  Matches ({row['num_matches']}):")
    
    for match_id in row['matches'][:3]:  # Show first 3 matches
        # Determine source
        if match_id.startswith('S2-'):
            match_rec = train_s2[train_s2['entity_id'] == match_id]
        else:
            match_rec = train_s3[train_s3['entity_id'] == match_id]
        
        if len(match_rec) > 0:
            match_rec = match_rec.iloc[0]
            print(f"    [{match_id}] {match_rec['business_name']}")
            print(f"      Address: {match_rec['business_address']}")
    print("-" * 100)

## 6. Noise Pattern Observations

In [ ]:
# Document observed noise patterns
print("\nCommon Noise Patterns to Handle:")
print("=" * 60)
print("\nBusiness Name Variations:")
print("  - Legal suffix differences (Inc/Incorporated, Pvt/Private, Ltd/Limited)")
print("  - Punctuation (& vs and, . vs no period)")
print("  - Case differences")
print("  - Abbreviations and acronyms")
print("  - Word order transpositions")
print("\nAddress Variations:")
print("  - Street type abbreviations (Rd/Road, St/Street)")
print("  - Missing/partial components (no ZIP, no state)")
print("  - Landmark-based descriptions")
print("  - Number formatting differences")
print("  - Component reordering")

## 7. Sample Singletons

In [ ]:
# Sample singleton entities
singletons = train_gt[train_gt['num_matches'] == 0].sample(min(10, len(train_gt[train_gt['num_matches'] == 0])))

print("\nSample Singleton Entities (No Matches):")
print("=" * 100)

for idx, row in singletons.head(10).iterrows():
    s1_id = row['source1_entity_id']
    s1_rec = train_s1[train_s1['entity_id'] == s1_id].iloc[0]
    
    print(f"[{s1_id}] {s1_rec['business_name']}")
    print(f"  Address: {s1_rec['business_address']}")
    print(f"  Country: {s1_rec['country']}")
    print()

## 8. Summary Statistics

In [ ]:
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

print("\nDataset Sizes:")
print(f"  Training:")
print(f"    Source 1: {len(train_s1):,} entities")
print(f"    Source 2: {len(train_s2):,} entities")
print(f"    Source 3: {len(train_s3):,} entities")
print(f"    Total: {len(train_all):,} entities")
print(f"\n  Test:")
print(f"    Source 1: {len(test_s1):,} entities")
print(f"    Source 2: {len(test_s2):,} entities")
print(f"    Source 3: {len(test_s3):,} entities")
print(f"    Total: {len(test_all):,} entities")

print(f"\nCountries:")
print(f"  Training: {sorted(train_all['country'].unique())}")
print(f"  Test: {sorted(test_all['country'].unique())}")

print(f"\nGround Truth:")
print(f"  Singletons: {(train_gt['num_matches'] == 0).sum():,} ({singleton_pct:.1f}%)")
print(f"  With matches: {(train_gt['num_matches'] > 0).sum():,} ({100-singleton_pct:.1f}%)")
print(f"  Avg matches (non-singleton): {train_gt[train_gt['num_matches'] > 0]['num_matches'].mean():.2f}")
print(f"  Max matches: {train_gt['num_matches'].max()}")

print("\n" + "=" * 80)
print("✓ EDA Complete")
print("=" * 80)